# Multi-modal Fine-Tuning for Question Paper Generation

This notebook uses the `unsloth` library to fine-tune a Vision-Language Model on your specific historical question papers dataset, including embedded images and diagrams.

In [ ]:
!git clone https://github.com/Prince649294u83/QP.git
%cd QP/dsatm-qpgen-backend

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastVisionModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit",
    load_in_4bit = load_in_4bit,
    use_gradient_checkpointing = "unsloth",
)

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision = True, # Finetune the vision encoder
    finetune_language = True, # Finetune the language model
    finetune_attention_modules = True, # Finetune attention
    finetune_mlp_modules = True, # Finetune MLP
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
from datasets import load_dataset
from PIL import Image
import os
from unsloth import standardize_sharegpt

# Note: You must have generated dataset.jsonl using the export_qp_dataset.py script locally before running this cell.
dataset = load_dataset("json", data_files="dataset.jsonl", split="train")

def format_sharegpt(example):
    imgs = example.get("images", [])
    has_image = False
    
    if imgs and len(imgs) > 0 and os.path.exists(imgs[0]):
        has_image = True
            
    # Standard ShareGPT format
    if has_image:
        human_value = "<image>\n" + example["instruction"]
    else:
        human_value = example["instruction"]
        
    conversations = [
        {"from": "human", "value": human_value},
        {"from": "gpt", "value": example["output"]}
    ]
        
    return { "conversations" : conversations }

def load_images_list(example):
    imgs = example.get("images", [])
    images_list = []
    if imgs and len(imgs) > 0 and os.path.exists(imgs[0]):
        try:
            # Unsloth standardize_sharegpt expects 'images' to be a list of PIL Images if there are images
            images_list = [Image.open(imgs[0]).convert("RGB")]
        except:
            pass
    return {"images": images_list}

# Map to ShareGPT and load images
dataset = dataset.map(format_sharegpt, batched = False)
dataset = dataset.map(load_images_list, batched = False)

# Let Unsloth handle the complex Llama 3.2 Vision schema conversion natively!
# This strictly avoids Hugging Face dataset schema inference bugs that cause KeyError: 0
dataset = standardize_sharegpt(dataset)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
import unsloth

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Packing is not supported for vision models yet
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        remove_unused_columns=False,
        dataset_kwargs = {"skip_prepare_dataset": True},
    ),
    data_collator = unsloth.UnslothVisionDataCollator(model, tokenizer),
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# Export to GGUF for local usage (e.g. Ollama)
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")